# Neural Style Transfer — GPU Training on Kaggle
**AdaIN + WCT (Adaptive Instance Normalization / Whitening-Coloring Transform)**

This notebook:
1. Verifies the GPU and installs dependencies
2. Defines the full model (encoder, decoder, AdaIN, WCT)
3. Generates VGG normalised weights from torchvision (no upload needed)
4. Trains the decoder with mixed-precision on GPU
5. Runs a quick inference test and saves outputs for download

**Before running:** Add your content and style image folders as Kaggle Datasets,
then set `CONTENT_DIR` and `STYLE_DIR` in Cell 2.

In [ ]:
# ── Cell 1: Install & verify GPU ──────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torchvision', 'Pillow', 'tqdm'], check=True)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

In [ ]:
# ── Cell 2: CONFIG — update dataset paths ─────────────────────────────────────
from pathlib import Path

# ▼▼▼ Set these to your Kaggle dataset paths ▼▼▼
CONTENT_DIR = '/kaggle/input/your-content-dataset/train2017'   # e.g. MS-COCO
STYLE_DIR   = '/kaggle/input/your-style-dataset/train'         # e.g. WikiArt
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

# Style transfer mode: 'adain' (fast) or 'wct' (richer, slower)
MODE = 'adain'

WORK_DIR   = Path('/kaggle/working')
VGG_PATH   = str(WORK_DIR / 'vgg_normalised.pth')
EXPERIMENT = f'nst_{MODE}_gpu'
SAVE_DIR   = WORK_DIR / 'experiment' / EXPERIMENT
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters — tuned for Kaggle T4 / P100 (16 GB VRAM)
IMAGE_SIZE     = 256
BATCH_SIZE     = 32    # raise to 64 on A100, lower to 16 for WCT mode
EPOCHS         = 20
LR             = 1e-4
ALPHA          = 1.0   # style strength: 0 = content only, 1 = full style
CONTENT_WEIGHT = 1.0
STYLE_WEIGHT   = 10.0
SAVE_EVERY     = 5
NUM_WORKERS    = 4

print(f'Mode     : {MODE}')
print(f'Content  : {CONTENT_DIR}')
print(f'Style    : {STYLE_DIR}')
print(f'Save     : {SAVE_DIR}')
print(f'Batch    : {BATCH_SIZE}  |  Epochs: {EPOCHS}  |  Size: {IMAGE_SIZE}px')

In [ ]:
# ── Cell 3: Full model — VGGEncoder, Decoder, AdaIN, WCT, NSTModel ────────────
import torch
import torch.nn as nn

vgg_arch = nn.Sequential(
    nn.Conv2d(3, 3, (1, 1)),                                              # 0  normalisation
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 1
    nn.Conv2d(3, 64, (3, 3)),                                             # 2
    nn.ReLU(),                                                            # 3  relu1_1
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 4
    nn.Conv2d(64, 64, (3, 3)),                                            # 5
    nn.ReLU(),                                                            # 6  relu1_2
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),                 # 7
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 8
    nn.Conv2d(64, 128, (3, 3)),                                           # 9
    nn.ReLU(),                                                            # 10 relu2_1
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 11
    nn.Conv2d(128, 128, (3, 3)),                                          # 12
    nn.ReLU(),                                                            # 13 relu2_2
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),                 # 14
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 15
    nn.Conv2d(128, 256, (3, 3)),                                          # 16
    nn.ReLU(),                                                            # 17 relu3_1
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 18
    nn.Conv2d(256, 256, (3, 3)), nn.ReLU(),                               # 19-20
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 21
    nn.Conv2d(256, 256, (3, 3)), nn.ReLU(),                               # 22-23
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 24
    nn.Conv2d(256, 256, (3, 3)), nn.ReLU(),                               # 25-26
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),                 # 27
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 28
    nn.Conv2d(256, 512, (3, 3)),                                          # 29
    nn.ReLU(),                                                            # 30 relu4_1
    nn.ReflectionPad2d((1, 1, 1, 1)), nn.Conv2d(512, 512, (3, 3)), nn.ReLU(),  # 31-33 relu4_2
    nn.ReflectionPad2d((1, 1, 1, 1)), nn.Conv2d(512, 512, (3, 3)), nn.ReLU(),  # 34-36 relu4_3
    nn.ReflectionPad2d((1, 1, 1, 1)), nn.Conv2d(512, 512, (3, 3)), nn.ReLU(),  # 37-39 relu4_4
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),                 # 40
    nn.ReflectionPad2d((1, 1, 1, 1)),                                     # 41
    nn.Conv2d(512, 512, (3, 3)),                                          # 42
    nn.ReLU(),                                                            # 43 relu5_1
)


class VGGEncoder(nn.Module):
    def __init__(self, vgg_path):
        super().__init__()
        vgg = vgg_arch
        vgg.load_state_dict(torch.load(vgg_path, map_location='cpu', weights_only=True))
        for p in vgg.parameters():
            p.requires_grad = False
        self.slice1 = vgg[:4]     # → relu1_1
        self.slice2 = vgg[4:11]   # → relu2_1
        self.slice3 = vgg[11:18]  # → relu3_1
        self.slice4 = vgg[18:31]  # → relu4_1
        self.slice5 = vgg[31:44]  # → relu5_1

    def forward(self, x, return_intermediates=False):
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        if return_intermediates:
            h5 = self.slice5(h4)
            return h1, h2, h3, h4, h5
        return h4


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(512, 256, (3,3)), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(256, 256, (3,3)), nn.ReLU(),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(256, 256, (3,3)), nn.ReLU(),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(256, 256, (3,3)), nn.ReLU(),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(256, 128, (3,3)), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(128, 128, (3,3)), nn.ReLU(),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d(128,  64, (3,3)), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d( 64,  64, (3,3)), nn.ReLU(),
            nn.ReflectionPad2d((1,1,1,1)), nn.Conv2d( 64,   3, (3,3)),
        )

    def forward(self, x):
        return self.net(x)


def adain(content_feat, style_feat):
    size = content_feat.shape
    s_mean = style_feat.view(size[0], size[1], -1).mean(2).view(size[0], size[1], 1, 1)
    s_std  = style_feat.view(size[0], size[1], -1).std(2).view(size[0], size[1], 1, 1) + 1e-5
    c_mean = content_feat.view(size[0], size[1], -1).mean(2).view(size[0], size[1], 1, 1)
    c_std  = content_feat.view(size[0], size[1], -1).std(2).view(size[0], size[1], 1, 1) + 1e-5
    return (content_feat - c_mean) / c_std * s_std + s_mean


def wct(content_feat, style_feat, alpha=1.0, eps=1e-5):
    """Whitening and Coloring Transform — full covariance matching via SVD.
    Processes each batch element independently (SVD is not batchable)."""
    b, c, h, w = content_feat.shape
    results = []
    for i in range(b):
        cf = content_feat[i].view(c, h * w).float()   # fp32 for numerical stability
        sf = style_feat[i].view(c, -1).float()

        c_mean = cf.mean(dim=1, keepdim=True)
        cf_c   = cf - c_mean
        c_cov  = cf_c @ cf_c.t() / (h * w - 1) + eps * torch.eye(c, device=cf.device)
        Uc, Sc, _ = torch.linalg.svd(c_cov)
        Dc = torch.diag(Sc.clamp(min=0).sqrt().reciprocal().clamp(max=1e4))
        whitened = Uc @ Dc @ Uc.t() @ cf_c            # U Λ^{-½} Uᵀ x

        s_mean = sf.mean(dim=1, keepdim=True)
        sf_c   = sf - s_mean
        s_cov  = sf_c @ sf_c.t() / (sf.shape[1] - 1) + eps * torch.eye(c, device=sf.device)
        Us, Ss, _ = torch.linalg.svd(s_cov)
        Ds = torch.diag(Ss.clamp(min=0).sqrt())
        colored = Us @ Ds @ Us.t() @ whitened + s_mean # U Λ^{½} Uᵀ x + μ_s

        result = alpha * colored + (1 - alpha) * cf
        results.append(result.to(content_feat.dtype).view(c, h, w))

    return torch.stack(results)


class NSTModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    @staticmethod
    def _mean_std(feat):
        b, c = feat.shape[:2]
        flat = feat.view(b, c, -1)
        return flat.mean(2).view(b, c, 1, 1), flat.std(2).view(b, c, 1, 1) + 1e-5

    def _style_loss(self, gen_feats, style_feats):
        loss = 0.0
        for gf, sf in zip(gen_feats, style_feats):
            gm, gs = self._mean_std(gf)
            sm, ss = self._mean_std(sf)
            loss += nn.functional.mse_loss(gm, sm) + nn.functional.mse_loss(gs, ss)
        return loss

    def forward(self, content, style, alpha=1.0,
                content_weight=1.0, style_weight=10.0, mode='adain'):
        style_feats  = self.encoder(style,   return_intermediates=True)
        content_feat = self.encoder(content, return_intermediates=False)

        if mode == 'adain':
            t = adain(content_feat, style_feats[3])
            t = alpha * t + (1 - alpha) * content_feat
        elif mode == 'wct':
            t = wct(content_feat, style_feats[3], alpha=alpha)
        else:
            raise ValueError(f"Unknown mode '{mode}'. Choose 'adain' or 'wct'.")

        generated = self.decoder(t)
        gen_feats = self.encoder(generated, return_intermediates=True)

        loss_c = nn.functional.mse_loss(gen_feats[3], t)
        loss_s = self._style_loss(gen_feats, style_feats)
        return generated, content_weight * loss_c, style_weight * loss_s


print('Model classes defined.')

In [ ]:
# ── Cell 4: Build vgg_normalised.pth from torchvision (no upload needed) ──────
from torchvision import models

MEAN = torch.tensor([0.485, 0.456, 0.406])
STD  = torch.tensor([0.229, 0.224, 0.225])

print('Downloading pretrained VGG19 from torchvision...')
vgg19 = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features

norm_conv = nn.Conv2d(3, 3, (1, 1))
norm_conv.weight.data = torch.diag(1.0 / STD).view(3, 3, 1, 1)
norm_conv.bias.data   = -MEAN / STD

# Map torchvision VGG19 layer indices → our vgg_arch indices
vgg19_to_ours = {
    0: 2,  2: 5,  5: 9,  7: 12,
    10: 16, 12: 19, 14: 22, 16: 25,
    19: 29, 21: 32, 23: 35, 25: 38,
    28: 42,
}

state = vgg_arch.state_dict()
state['0.weight'] = norm_conv.weight.data
state['0.bias']   = norm_conv.bias.data
for v_idx, o_idx in vgg19_to_ours.items():
    state[f'{o_idx}.weight'] = vgg19[v_idx].weight.data.clone()
    state[f'{o_idx}.bias']   = vgg19[v_idx].bias.data.clone()

vgg_arch.load_state_dict(state)
torch.save(state, VGG_PATH)
print(f'Saved vgg_normalised.pth  ({Path(VGG_PATH).stat().st_size / 1e6:.1f} MB)')

In [ ]:
# ── Cell 5: Dataset & DataLoader ──────────────────────────────────────────────
import random
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None          # disable decompression bomb limit

EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def gather_images(directory):
    return sorted(f for f in Path(directory).rglob('*')
                  if f.is_file() and f.suffix.lower() in EXTENSIONS)

def build_transform(size, augment=True):
    ops = [transforms.Resize(int(size * 1.1))]
    if augment:
        ops += [
            transforms.RandomCrop(size),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        ]
    else:
        ops.append(transforms.CenterCrop(size))
    ops.append(transforms.ToTensor())
    return transforms.Compose(ops)

class StyleTransferDataset(Dataset):
    def __init__(self, content_dir, style_dir, image_size=256, augment=True):
        self.content_paths = gather_images(content_dir)
        self.style_paths   = gather_images(style_dir)
        if not self.content_paths:
            raise FileNotFoundError(f'No images found in {content_dir}')
        if not self.style_paths:
            raise FileNotFoundError(f'No images found in {style_dir}')
        self.transform = build_transform(image_size, augment)
        print(f'Content : {len(self.content_paths):,}  |  Style: {len(self.style_paths):,}')

    def __len__(self):
        return len(self.content_paths)

    def __getitem__(self, idx):
        content = Image.open(self.content_paths[idx]).convert('RGB')
        style   = Image.open(
            self.style_paths[random.randint(0, len(self.style_paths) - 1)]
        ).convert('RGB')
        return self.transform(content), self.transform(style)

dataset = StyleTransferDataset(CONTENT_DIR, STYLE_DIR, IMAGE_SIZE, augment=True)
loader  = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(NUM_WORKERS > 0),
)
print(f'Steps per epoch : {len(loader):,}')

In [ ]:
# ── Cell 6: Training (mixed-precision for AdaIN, fp32 for WCT) ────────────────
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from torchvision.utils import save_image
from tqdm.auto import tqdm
import time

encoder = VGGEncoder(VGG_PATH).to(DEVICE)
decoder = Decoder().to(DEVICE)
model   = NSTModel(encoder, decoder).to(DEVICE)

optimizer = Adam(decoder.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Mixed precision only for AdaIN on CUDA (WCT SVD is numerically sensitive in fp16)
use_amp = (DEVICE.type == 'cuda') and (MODE == 'adain')
scaler  = GradScaler('cuda', enabled=use_amp)
print(f'Mixed precision : {use_amp}')

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    total_c = total_s = 0.0
    t0 = time.time()

    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for content, style in pbar:
        content, style = content.to(DEVICE), style.to(DEVICE)

        with autocast('cuda', enabled=use_amp):
            generated, loss_c, loss_s = model(
                content, style,
                alpha=ALPHA,
                content_weight=CONTENT_WEIGHT,
                style_weight=STYLE_WEIGHT,
                mode=MODE,
            )
            loss = loss_c + loss_s

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_c += loss_c.item()
        total_s += loss_s.item()
        pbar.set_postfix(c=f'{loss_c.item():.3f}', s=f'{loss_s.item():.3f}')

    scheduler.step()
    avg_c     = total_c / len(loader)
    avg_s     = total_s / len(loader)
    avg_total = avg_c + avg_s
    epoch_min = (time.time() - t0) / 60

    if DEVICE.type == 'cuda':
        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        mem_str = f'  VRAM: {vram_gb:.1f}GB'
    else:
        mem_str = ''

    print(f'Epoch {epoch+1:>2}/{EPOCHS} | '
          f'Content: {avg_c:.4f} | Style: {avg_s:.4f} | '
          f'Total: {avg_total:.4f} | {epoch_min:.1f}min{mem_str}')

    if avg_total < best_loss:
        best_loss = avg_total
        torch.save(decoder.state_dict(), str(SAVE_DIR / 'decoder_best.pth'))
        print(f'  ↑ New best {best_loss:.4f} → decoder_best.pth')

    if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
        ckpt_path = SAVE_DIR / f'decoder_epoch_{epoch+1:04d}.pth'
        torch.save({
            'epoch': epoch,
            'decoder': decoder.state_dict(),
            'optimizer': optimizer.state_dict(),
            'loss_c': avg_c, 'loss_s': avg_s,
            'mode': MODE,
        }, str(ckpt_path))
        sample = torch.cat(
            [content[:4], style[:4], generated[:4].detach().clamp(0, 1)], dim=0
        )
        save_image(sample, str(SAVE_DIR / f'samples_epoch_{epoch+1:04d}.png'),
                   nrow=4, normalize=False)
        print(f'  Checkpoint + samples → {ckpt_path.name}')

torch.save(decoder.state_dict(), str(SAVE_DIR / 'decoder_final.pth'))
print(f'\nDone. decoder_final.pth saved to {SAVE_DIR}')

In [ ]:
# ── Cell 7: Quick inference test ──────────────────────────────────────────────
import matplotlib.pyplot as plt

to_tensor = transforms.Compose([
    transforms.Resize(512), transforms.CenterCrop(512), transforms.ToTensor()
])

c_img = Image.open(dataset.content_paths[0]).convert('RGB')
s_img = Image.open(dataset.style_paths[0]).convert('RGB')

c = to_tensor(c_img).unsqueeze(0).to(DEVICE)
s = to_tensor(s_img).unsqueeze(0).to(DEVICE)

decoder.eval()
encoder.eval()
with torch.no_grad():
    cf          = encoder(c)
    sf          = encoder(s, return_intermediates=True)
    if MODE == 'adain':
        t = adain(cf, sf[3])
        t = ALPHA * t + (1 - ALPHA) * cf
    else:
        t = wct(cf, sf[3], alpha=ALPHA)
    out = decoder(t).clamp(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title in zip(axes,
        [c.squeeze().cpu(), s.squeeze().cpu(), out.squeeze().cpu()],
        ['Content', 'Style', f'Generated ({MODE.upper()})']):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=16)
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'inference_test.png'), dpi=150)
plt.show()
print('Inference test saved.')

In [ ]:
# ── Cell 8: List output files for download ────────────────────────────────────
print('Files ready in /kaggle/working/:\n')
for f in sorted(SAVE_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(WORK_DIR)):<55} {f.stat().st_size/1e6:>6.1f} MB')

print('\nTo use locally:')
print('  1. Download decoder_final.pth (or decoder_best.pth)')
print('  2. Place it at  experiment/<name>/decoder_final.pth')
print('  3. Reload the app — AdaIN and WCT modes are ready.')